### Import

In [1]:
import os
import sys
import re
import numpy as np
import pandas as pd
import datetime as dt
from tqdm import tqdm
from matplotlib import pyplot as plt 

from multiprocessing import Pool
from multiprocessing import cpu_count

pd.set_option('display.max_columns', 500)

### General parameters

In [2]:
path_timeseries = "../Data/Output"

### List of variables

In [3]:
all_variables = []

with open("../Data/csvExtract/variables.txt", "r") as f:
    for variable in f:
        all_variables.append(variable.strip())
        
len(all_variables)

201

### Read csv files

In [4]:
def dataframe_from_csv(path, header=0, index_col=False):
    return pd.read_csv(path, header=header, index_col=index_col)

In [5]:
def filter_on_variabels(all_tables, all_variables):
    all_tables = all_tables[all_tables['itemname'].isin(all_variables)]
    return all_tables

### Outlier detection

In [6]:
def outlier_removal(all_series):
    
    column = list(all_series.columns)

    for col in column:
        if col not in ['patientunitstayid', 'itemoffset', 'ICD-9', 'ICD-10', 'ICD-10_Embedding',
                       'pastHistory', 'elixhauser_comorbidity', 'Delirium Scale', 'O2 Admin Device']:
            try:
                all_series[col] = all_series[col].astype(float)
            except:
                continue
                
    all_series.loc[all_series['Urine_IO'] < 0 ,     'Urine_IO'] = np.nan
    all_series.loc[all_series['Urine_IO'] > 1200 ,  'Urine_IO'] = np.nan
    all_series.loc[all_series['Propofol_IO'] < 0 ,    'Propofol_IO'] = np.nan
    all_series.loc[all_series['Propofol_IO'] > 400 ,  'Propofol_IO'] = np.nan
    all_series.loc[all_series['Fentanyl_IO'] < 0 ,   'Fentanyl_IO'] = np.nan
    all_series.loc[all_series['Fentanyl_IO'] > 250 , 'Fentanyl_IO'] = np.nan
    all_series.loc[all_series['Insulin_IO'] < 0 ,   'Insulin_IO'] = np.nan
    all_series.loc[all_series['Insulin_IO'] > 150 , 'Insulin_IO'] = np.nan
    all_series.loc[all_series['Heparin_IO'] < 0 ,   'Heparin_IO'] = np.nan
    all_series.loc[all_series['Heparin_IO'] > 500 , 'Heparin_IO'] = np.nan
    all_series.loc[all_series['Midazolam_IO'] < 0 ,   'Midazolam_IO'] = np.nan
    all_series.loc[all_series['Midazolam_IO'] > 120 , 'Midazolam_IO'] = np.nan
    all_series.loc[all_series['Dexmedetomidine_IO'] < 0 ,   'Dexmedetomidine_IO'] = np.nan
    all_series.loc[all_series['Dexmedetomidine_IO'] > 300 , 'Dexmedetomidine_IO'] = np.nan
    all_series.loc[all_series['Vassopressin_IO'] < 0 ,   'Vassopressin_IO'] = np.nan
    all_series.loc[all_series['Vassopressin_IO'] > 100 , 'Vassopressin_IO'] = np.nan
    all_series.loc[all_series['Albumin_IO'] < 0 ,    'Albumin_IO'] = np.nan
    all_series.loc[all_series['Albumin_IO'] > 1500 , 'Albumin_IO'] = np.nan
    all_series.loc[all_series['Ceftriaxone_IO'] < 0 ,    'Ceftriaxone_IO'] = np.nan
    all_series.loc[all_series['Ceftriaxone_IO'] > 500 ,  'Ceftriaxone_IO'] = np.nan
    all_series.loc[all_series['Cefazolin_IO'] < 0 ,     'Cefazolin_IO'] = np.nan
    all_series.loc[all_series['Cefazolin_IO'] > 1100 ,  'Cefazolin_IO'] = np.nan
    all_series.loc[all_series['Cefepime_IO'] < 0 ,     'Cefepime_IO'] = np.nan
    all_series.loc[all_series['Cefepime_IO'] > 1100 ,  'Cefepime_IO'] = np.nan
    all_series.loc[all_series['Ceftazidime_IO'] < 0 ,     'Ceftazidime_IO'] = np.nan
    all_series.loc[all_series['Ceftazidime_IO'] > 200 ,   'Ceftazidime_IO'] = np.nan
    all_series.loc[all_series['Vancomycin_IO'] < 0 ,     'Vancomycin_IO'] = np.nan
    all_series.loc[all_series['Vancomycin_IO'] > 1100 ,  'Vancomycin_IO'] = np.nan
    all_series.loc[all_series['Clindamycin_IO'] < 0 ,    'Clindamycin_IO'] = np.nan
    all_series.loc[all_series['Clindamycin_IO'] > 500 ,  'Clindamycin_IO'] = np.nan
    all_series.loc[all_series['Metronidazole_IO'] < 0 ,    'Metronidazole_IO'] = np.nan
    all_series.loc[all_series['Metronidazole_IO'] > 550 ,  'Metronidazole_IO'] = np.nan
    all_series.loc[all_series['Meropenem_IO'] < 0 ,    'Meropenem_IO'] = np.nan
    all_series.loc[all_series['Meropenem_IO'] > 550 ,  'Meropenem_IO'] = np.nan
    all_series.loc[all_series['Acyclovir_IO'] < 0 ,    'Acyclovir_IO'] = np.nan
    all_series.loc[all_series['Acyclovir_IO'] > 400 ,  'Acyclovir_IO'] = np.nan
    all_series.loc[all_series['Azithromycin_IO'] < 0 ,     'Azithromycin_IO'] = np.nan
    all_series.loc[all_series['Azithromycin_IO'] > 1050 ,  'Azithromycin_IO'] = np.nan
    all_series.loc[all_series['Levofloxacin_IO'] < 0 ,    'Levofloxacin_IO'] = np.nan
    all_series.loc[all_series['Levofloxacin_IO'] > 750 ,  'Levofloxacin_IO'] = np.nan
    all_series.loc[all_series['Micafungin_IO'] < 0 ,    'Micafungin_IO'] = np.nan
    all_series.loc[all_series['Micafungin_IO'] > 550 ,  'Micafungin_IO'] = np.nan
    all_series.loc[all_series['Fluconazole_IO'] < 0 ,    'Fluconazole_IO'] = np.nan
    all_series.loc[all_series['Fluconazole_IO'] > 750 ,  'Fluconazole_IO'] = np.nan
    all_series.loc[all_series['Thiamine_IO'] < 0 ,    'Thiamine_IO'] = np.nan
    all_series.loc[all_series['Thiamine_IO'] > 800 ,  'Thiamine_IO'] = np.nan
    all_series.loc[all_series['Dobutamine_IO'] < 0 ,     'Dobutamine_IO'] = np.nan
    all_series.loc[all_series['Dobutamine_IO'] > 400 ,   'Dobutamine_IO'] = np.nan
    all_series.loc[all_series['Milrinone_IO'] < 0 ,    'Milrinone_IO'] = np.nan
    all_series.loc[all_series['Milrinone_IO'] > 150 ,  'Milrinone_IO'] = np.nan
    all_series.loc[all_series['Fluids_IO'] < 0 ,     'Fluids_IO'] = np.nan
    all_series.loc[all_series['Fluids_IO'] > 1200 ,  'Fluids_IO'] = np.nan
    all_series.loc[all_series['OralIntake_IO'] < 0 ,     'OralIntake_IO'] = np.nan
    all_series.loc[all_series['OralIntake_IO'] > 1200 ,  'OralIntake_IO'] = np.nan
    all_series.loc[all_series['P.O._IO'] < 0 ,     'P.O._IO'] = np.nan
    all_series.loc[all_series['P.O._IO'] > 1000 ,  'P.O._IO'] = np.nan
    all_series.loc[all_series['SodiumChloride_IO'] < 0 ,    'SodiumChloride_IO'] = np.nan
    all_series.loc[all_series['SodiumChloride_IO'] > 600 ,  'SodiumChloride_IO'] = np.nan
    all_series.loc[all_series['IVPB_IO'] < 0 ,    'IVPB_IO'] = np.nan
    all_series.loc[all_series['IVPB_IO'] > 750 ,  'IVPB_IO'] = np.nan
    all_series.loc[all_series['Stool_IO'] < 0 ,     'Stool_IO'] = np.nan
    all_series.loc[all_series['Stool_IO'] > 1100 ,  'Stool_IO'] = np.nan
    all_series.loc[all_series['Crystalloids_IO'] < 0 ,    'Crystalloids_IO'] = np.nan
    all_series.loc[all_series['Crystalloids_IO'] > 700 ,  'Crystalloids_IO'] = np.nan
    all_series.loc[all_series['NSIVF_IO'] < 0 ,    'NSIVF_IO'] = np.nan
    all_series.loc[all_series['NSIVF_IO'] > 300 ,  'NSIVF_IO'] = np.nan
    all_series.loc[all_series['Norepinephrine_IO'] < 0 ,    'Norepinephrine_IO'] = np.nan
    all_series.loc[all_series['Norepinephrine_IO'] > 500 ,  'Norepinephrine_IO'] = np.nan
    all_series.loc[all_series['Amiodarone_IO'] < 0 ,    'Amiodarone_IO'] = np.nan
    all_series.loc[all_series['Amiodarone_IO'] > 350 ,  'Amiodarone_IO'] = np.nan
    all_series.loc[all_series['Phenylephrine_IO'] < 0 ,    'Phenylephrine_IO'] = np.nan
    all_series.loc[all_series['Phenylephrine_IO'] > 350 ,  'Phenylephrine_IO'] = np.nan
    all_series.loc[all_series['Epinephrine_IO'] < 0 ,    'Epinephrine_IO'] = np.nan
    all_series.loc[all_series['Epinephrine_IO'] > 350 ,  'Epinephrine_IO'] = np.nan
    all_series.loc[all_series['Nicardipine_IO'] < 0 ,     'Nicardipine_IO'] = np.nan
    all_series.loc[all_series['Nicardipine_IO'] > 1200 ,  'Nicardipine_IO'] = np.nan
    all_series.loc[all_series['Pantoprazole_IO'] < 0 ,    'Pantoprazole_IO'] = np.nan
    all_series.loc[all_series['Pantoprazole_IO'] > 200 ,  'Pantoprazole_IO'] = np.nan
    all_series.loc[all_series['Diltiazem_IO'] < 0 ,    'Diltiazem_IO'] = np.nan
    all_series.loc[all_series['Diltiazem_IO'] > 250 ,  'Diltiazem_IO'] = np.nan
    all_series.loc[all_series['Nitroglycerin_IO'] < 0 ,    'Nitroglycerin_IO'] = np.nan
    all_series.loc[all_series['Nitroglycerin_IO'] > 350 ,  'Nitroglycerin_IO'] = np.nan
    all_series.loc[all_series['Bodyweight'] < 10 ,    'Bodyweight'] = np.nan
    all_series.loc[all_series['Bodyweight'] > 260 ,   'Bodyweight'] = np.nan
    all_series.loc[all_series['Non-Invasive BP Mean'] < 10 ,    'Non-Invasive BP Mean'] = np.nan
    all_series.loc[all_series['Non-Invasive BP Mean'] > 200 ,   'Non-Invasive BP Mean'] = np.nan
    all_series.loc[all_series['Non-Invasive BP Systolic'] < 10 ,    'Non-Invasive BP Systolic'] = np.nan
    all_series.loc[all_series['Non-Invasive BP Systolic'] > 250 ,   'Non-Invasive BP Systolic'] = np.nan
    all_series.loc[all_series['Non-Invasive BP Diastolic'] < 10 ,    'Non-Invasive BP Diastolic'] = np.nan
    all_series.loc[all_series['Non-Invasive BP Diastolic'] > 200 ,   'Non-Invasive BP Diastolic'] = np.nan
    all_series.loc[all_series['Invasive BP Mean'] < 10 ,    'Invasive BP Mean'] = np.nan
    all_series.loc[all_series['Invasive BP Mean'] > 200 ,   'Invasive BP Mean'] = np.nan
    all_series.loc[all_series['Invasive BP Systolic'] < 10 ,    'Invasive BP Systolic'] = np.nan
    all_series.loc[all_series['Invasive BP Systolic'] > 250 ,   'Invasive BP Systolic'] = np.nan
    all_series.loc[all_series['Invasive BP Diastolic'] < 10 ,    'Invasive BP Diastolic'] = np.nan
    all_series.loc[all_series['Invasive BP Diastolic'] > 200 ,   'Invasive BP Diastolic'] = np.nan
    all_series.loc[all_series['PA Mean'] < 1 ,    'PA Mean'] = np.nan
    all_series.loc[all_series['PA Mean'] > 100 ,  'PA Mean'] = np.nan
    all_series.loc[all_series['PA Systolic'] < 1 ,    'PA Systolic'] = np.nan
    all_series.loc[all_series['PA Systolic'] > 140 ,  'PA Systolic'] = np.nan
    all_series.loc[all_series['PA Diastolic'] < 1 ,    'PA Diastolic'] = np.nan
    all_series.loc[all_series['PA Diastolic'] > 80 ,  'PA Diastolic'] = np.nan
    all_series.loc[all_series['Temperature (C)'] < 30 ,  'Temperature (C)'] = np.nan
    all_series.loc[all_series['Temperature (C)'] > 45 ,  'Temperature (C)'] = np.nan
    all_series.loc[all_series['Heart Rate'] < 20 ,    'Heart Rate'] = np.nan
    all_series.loc[all_series['Heart Rate'] > 200 ,   'Heart Rate'] = np.nan
    all_series.loc[all_series['Respiratory Rate'] < 1 ,   'Respiratory Rate'] = np.nan
    all_series.loc[all_series['Respiratory Rate'] > 60 ,  'Respiratory Rate'] = np.nan
    all_series.loc[all_series['CVP'] < 0 ,   'CVP'] = np.nan
    all_series.loc[all_series['CVP'] > 80 ,  'CVP'] = np.nan
    all_series.loc[all_series['ETCO2'] < 0 ,    'ETCO2'] = np.nan
    all_series.loc[all_series['ETCO2'] > 110 ,  'ETCO2'] = np.nan
    all_series.loc[all_series['PT'] < 0 ,   'PT'] = np.nan
    all_series.loc[all_series['PT'] > 80 ,  'PT'] = np.nan
    all_series.loc[all_series['PTT'] < 0 ,    'PTT'] = np.nan
    all_series.loc[all_series['PTT'] > 220 ,  'PTT'] = np.nan
    all_series.loc[all_series['PT - INR'] < 0 ,    'PT - INR'] = np.nan
    all_series.loc[all_series['PT - INR'] > 10 ,  'PT - INR'] = np.nan
    all_series.loc[all_series['pH'] < 6 ,  'pH'] = np.nan
    all_series.loc[all_series['pH'] > 8 ,  'pH'] = np.nan
    all_series.loc[all_series['lactate'] < 0 ,   'lactate'] = np.nan
    all_series.loc[all_series['lactate'] > 30 ,  'lactate'] = np.nan
    all_series.loc[all_series['LDH'] < 0 ,     'LDH'] = np.nan
    all_series.loc[all_series['LDH'] > 6000 ,  'LDH'] = np.nan
    all_series.loc[all_series['Base Excess'] < -40 ,  'Base Excess'] = np.nan
    all_series.loc[all_series['Base Excess'] > 40  ,  'Base Excess'] = np.nan
    all_series.loc[all_series['anion gap'] < 0 ,   'anion gap'] = np.nan
    all_series.loc[all_series['anion gap'] > 40 ,  'anion gap'] = np.nan
    all_series.loc[all_series['Bicarbonate'] < 0 ,   'Bicarbonate'] = np.nan
    all_series.loc[all_series['Bicarbonate'] > 60 ,  'Bicarbonate'] = np.nan
    all_series.loc[all_series['creatinine'] < 0 ,   'creatinine'] = np.nan
    all_series.loc[all_series['creatinine'] > 25 ,  'creatinine'] = np.nan
    all_series.loc[all_series['Hct'] < 5 ,   'Hct'] = np.nan
    all_series.loc[all_series['Hct'] > 75 ,  'Hct'] = np.nan
    all_series.loc[all_series['Hgb'] < 1 ,   'Hgb'] = np.nan
    all_series.loc[all_series['Hgb'] > 40 ,  'Hgb'] = np.nan
    all_series.loc[all_series['total bilirubin'] < 0 ,   'total bilirubin'] = np.nan
    all_series.loc[all_series['total bilirubin'] > 50 ,  'total bilirubin'] = np.nan
    all_series.loc[all_series['direct bilirubin'] < 0 ,   'direct bilirubin'] = np.nan
    all_series.loc[all_series['direct bilirubin'] > 50 ,  'direct bilirubin'] = np.nan
    all_series.loc[all_series['MPV'] < 0 ,   'MPV'] = np.nan
    all_series.loc[all_series['MPV'] > 20 ,  'MPV'] = np.nan
    all_series.loc[all_series['MCV'] < 10 ,   'MCV'] = np.nan
    all_series.loc[all_series['MCV'] > 150 ,  'MCV'] = np.nan
    all_series.loc[all_series['MCH'] < 10 ,  'MCH'] = np.nan
    all_series.loc[all_series['MCH'] > 50 ,  'MCH'] = np.nan
    all_series.loc[all_series['MCHC'] < 10 ,  'MCHC'] = np.nan
    all_series.loc[all_series['MCHC'] > 50 ,  'MCHC'] = np.nan
    all_series.loc[all_series['RDW'] < 10 ,  'RDW'] = np.nan
    all_series.loc[all_series['RDW'] > 40 ,  'RDW'] = np.nan
    all_series.loc[all_series['RBC'] < 0 ,   'RBC'] = np.nan
    all_series.loc[all_series['RBC'] > 10 ,  'RBC'] = np.nan
    all_series.loc[all_series['WBC x 1000'] < 0 ,    'WBC x 1000'] = np.nan
    all_series.loc[all_series['WBC x 1000'] > 70 ,   'WBC x 1000'] = np.nan
    all_series.loc[all_series['platelets x 1000'] < 0 ,     'platelets x 1000'] = np.nan
    all_series.loc[all_series['platelets x 1000'] > 1000 ,  'platelets x 1000'] = np.nan
    all_series.loc[all_series['Glucose'] < 0 ,    'Glucose'] = np.nan
    all_series.loc[all_series['Glucose'] > 900 ,  'Glucose'] = np.nan
    all_series.loc[all_series['ammonia'] < 0 ,    'ammonia'] = np.nan
    all_series.loc[all_series['ammonia'] > 500 ,  'ammonia'] = np.nan
    all_series.loc[all_series['magnesium'] < 0 ,   'magnesium'] = np.nan
    all_series.loc[all_series['magnesium'] > 10 ,  'magnesium'] = np.nan
    all_series.loc[all_series['phosphate'] < 0 ,    'phosphate'] = np.nan
    all_series.loc[all_series['phosphate'] > 15 ,   'phosphate'] = np.nan
    all_series.loc[all_series['alkaline phos.'] < 0 ,    'alkaline phos.'] = np.nan
    all_series.loc[all_series['alkaline phos.'] > 750 ,  'alkaline phos.'] = np.nan
    all_series.loc[all_series['potassium'] < 0 ,   'potassium'] = np.nan
    all_series.loc[all_series['potassium'] > 10 ,  'potassium'] = np.nan
    all_series.loc[all_series['sodium'] < 60 ,   'sodium'] = np.nan
    all_series.loc[all_series['sodium'] > 200 ,  'sodium'] = np.nan
    all_series.loc[all_series['chloride'] < 20 ,    'chloride'] = np.nan
    all_series.loc[all_series['chloride'] > 180 ,  'chloride'] = np.nan
    all_series.loc[all_series['calcium'] < 0 ,   'calcium'] = np.nan
    all_series.loc[all_series['calcium'] > 20 ,  'calcium'] = np.nan
    all_series.loc[all_series['ionized calcium'] < 0 ,  'ionized calcium'] = np.nan
    all_series.loc[all_series['ionized calcium'] > 9 ,  'ionized calcium'] = np.nan
    all_series.loc[all_series['total cholesterol'] < 0 ,    'total cholesterol'] = np.nan
    all_series.loc[all_series['total cholesterol'] > 500 ,  'total cholesterol'] = np.nan
    all_series.loc[all_series['paO2'] < 0 ,    'paO2'] = np.nan
    all_series.loc[all_series['paO2'] > 600 ,  'paO2'] = np.nan
    all_series.loc[all_series['paCO2'] < 0 ,    'paCO2'] = np.nan
    all_series.loc[all_series['paCO2'] > 175 ,  'paCO2'] = np.nan
    all_series.loc[all_series['ALT (SGPT)'] < 0 ,     'ALT (SGPT)'] = np.nan
    all_series.loc[all_series['ALT (SGPT)'] > 1200 ,  'ALT (SGPT)'] = np.nan
    all_series.loc[all_series['AST (SGOT)'] < 0 ,     'AST (SGOT)'] = np.nan
    all_series.loc[all_series['AST (SGOT)'] > 1200 ,  'AST (SGOT)'] = np.nan
    all_series.loc[all_series['-bands'] < 0 ,    '-bands'] = np.nan
    all_series.loc[all_series['-bands'] > 80 ,   '-bands'] = np.nan
    all_series.loc[all_series['-polys'] < 0 ,    '-polys'] = np.nan
    all_series.loc[all_series['-polys'] > 105 ,  '-polys'] = np.nan
    all_series.loc[all_series['amylase'] < 0 ,     'amylase'] = np.nan
    all_series.loc[all_series['amylase'] > 1000 ,  'amylase'] = np.nan
    all_series.loc[all_series['lipase'] < 0 ,     'lipase'] = np.nan
    all_series.loc[all_series['lipase'] > 2000 ,  'lipase'] = np.nan
    all_series.loc[all_series['-lymphs'] < 0 ,     '-lymphs'] = np.nan
    all_series.loc[all_series['-lymphs'] > 105 ,   '-lymphs'] = np.nan
    all_series.loc[all_series['-monos'] < 0 ,    '-monos'] = np.nan
    all_series.loc[all_series['-monos'] > 60 ,   '-monos'] = np.nan
    all_series.loc[all_series['-eos'] < 0 ,    '-eos'] = np.nan
    all_series.loc[all_series['-eos'] > 50 ,   '-eos'] = np.nan
    all_series.loc[all_series['-basos'] < 0 ,   '-basos'] = np.nan
    all_series.loc[all_series['-basos'] > 10 ,  '-basos'] = np.nan
    all_series.loc[all_series['LPM O2'] < 0 ,    'LPM O2'] = np.nan
    all_series.loc[all_series['LPM O2'] > 100 ,  'LPM O2'] = np.nan
    all_series.loc[all_series['O2 Content'] < 0 ,   'O2 Content'] = np.nan
    all_series.loc[all_series['O2 Content'] > 35 ,  'O2 Content'] = np.nan
    all_series.loc[all_series['O2 Saturation'] < 0 ,    'O2 Saturation'] = np.nan
    all_series.loc[all_series['O2 Saturation'] > 105 ,  'O2 Saturation'] = np.nan
    all_series.loc[all_series['Total CO2'] < 0 ,    'Total CO2'] = np.nan
    all_series.loc[all_series['Total CO2'] > 100 ,  'Total CO2'] = np.nan
    all_series.loc[all_series['albumin'] < 0 ,   'albumin'] = np.nan
    all_series.loc[all_series['albumin'] > 10 ,  'albumin'] = np.nan
    all_series.loc[all_series['troponin - T'] < 0 ,   'troponin - T'] = np.nan
    all_series.loc[all_series['troponin - T'] > 20 ,  'troponin - T'] = np.nan
    all_series.loc[all_series['troponin - I'] < 0 ,   'troponin - I'] = np.nan
    all_series.loc[all_series['troponin - I'] > 40 ,  'troponin - I'] = np.nan
    all_series.loc[all_series['Vancomycin - peak'] < 0 ,    'Vancomycin - peak'] = np.nan
    all_series.loc[all_series['Vancomycin - peak'] > 70 ,   'Vancomycin - peak'] = np.nan
    all_series.loc[all_series['Vancomycin - trough'] < 0 ,    'Vancomycin - trough'] = np.nan
    all_series.loc[all_series['Vancomycin - trough'] > 70 ,   'Vancomycin - trough'] = np.nan
    all_series.loc[all_series['Vancomycin - random'] < 0 ,    'Vancomycin - random'] = np.nan
    all_series.loc[all_series['Vancomycin - random'] > 70 ,   'Vancomycin - random'] = np.nan
    all_series.loc[all_series['triglycerides'] < 0 ,     'triglycerides'] = np.nan
    all_series.loc[all_series['triglycerides'] > 1200 ,  'triglycerides'] = np.nan
    all_series.loc[all_series['fibrinogen'] < 0 ,     'fibrinogen'] = np.nan
    all_series.loc[all_series['fibrinogen'] > 1200 ,  'fibrinogen'] = np.nan
    all_series.loc[all_series['transferrin'] < 0 ,    'transferrin'] = np.nan
    all_series.loc[all_series['transferrin'] > 550 ,  'transferrin'] = np.nan
    all_series.loc[all_series['Ferritin'] < 0 ,     'Ferritin'] = np.nan
    all_series.loc[all_series['Ferritin'] > 4000 ,  'Ferritin'] = np.nan
    all_series.loc[all_series['total protein'] < 0 ,    'total protein'] = np.nan
    all_series.loc[all_series['total protein'] > 20 ,   'total protein'] = np.nan
    all_series.loc[all_series['PEEP'] < 0 ,    'PEEP'] = np.nan
    all_series.loc[all_series['PEEP'] > 40 ,   'PEEP'] = np.nan
    all_series.loc[all_series['Tidal Volume'] < 0 ,     'Tidal Volume'] = np.nan
    all_series.loc[all_series['Tidal Volume'] > 1100 ,  'Tidal Volume'] = np.nan
    all_series.loc[all_series['Vent Rate'] < 0 ,   'Vent Rate'] = np.nan
    all_series.loc[all_series['Vent Rate'] > 75 ,  'Vent Rate'] = np.nan
    all_series.loc[all_series['FiO2'] < 0 ,    'FiO2'] = np.nan
    all_series.loc[all_series['FiO2'] > 105 ,  'FiO2'] = np.nan
    all_series.loc[all_series['BUN'] < 0 ,    'BUN'] = np.nan
    all_series.loc[all_series['BUN'] > 170 ,  'BUN'] = np.nan
    all_series.loc[all_series['TSH'] < 0 ,    'TSH'] = np.nan
    all_series.loc[all_series['TSH'] > 60 ,   'TSH'] = np.nan
    all_series.loc[all_series['Pressure Support'] < 0 ,    'Pressure Support'] = np.nan
    all_series.loc[all_series['Pressure Support'] > 70 ,   'Pressure Support'] = np.nan
    all_series.loc[all_series['Pressure Control'] < 0 ,   'Pressure Control'] = np.nan
    all_series.loc[all_series['Pressure Control'] > 80 ,  'Pressure Control'] = np.nan
    all_series.loc[all_series['Peak Airway/Pressure'] < 0 ,    'Peak Airway/Pressure'] = np.nan
    all_series.loc[all_series['Peak Airway/Pressure'] > 80 ,   'Peak Airway/Pressure'] = np.nan
    all_series.loc[all_series['Flow Rate'] < 0 ,    'Flow Rate'] = np.nan
    all_series.loc[all_series['Flow Rate'] > 20 ,   'Flow Rate'] = np.nan
    all_series.loc[all_series['SpO2'] < 10 ,   'SpO2'] = np.nan
    all_series.loc[all_series['SpO2'] > 105 ,  'SpO2'] = np.nan
    all_series.loc[all_series['SVO2'] < 0 ,    'SVO2'] = np.nan
    all_series.loc[all_series['SVO2'] > 105 ,  'SVO2'] = np.nan
    all_series.loc[all_series['Pulse'] < 0 ,    'Pulse'] = np.nan
    all_series.loc[all_series['Pulse'] > 200 ,  'Pulse'] = np.nan
    all_series.loc[all_series['MAP (mmHg)'] < 10 ,   'MAP (mmHg)'] = np.nan
    all_series.loc[all_series['MAP (mmHg)'] > 175 ,  'MAP (mmHg)'] = np.nan
    all_series.loc[all_series['Total Respiratory Rate'] < 1 ,    'Total Respiratory Rate'] = np.nan
    all_series.loc[all_series['Total Respiratory Rate'] > 60 ,   'Total Respiratory Rate'] = np.nan
    all_series.loc[all_series['Exhaled MV'] < 0 ,    'Exhaled MV'] = np.nan
    all_series.loc[all_series['Exhaled MV'] > 30 ,   'Exhaled MV'] = np.nan
    all_series.loc[all_series['Exhaled Vt'] < 100 ,    'Exhaled Vt'] = np.nan
    all_series.loc[all_series['Exhaled Vt'] > 1000 ,   'Exhaled Vt'] = np.nan
    all_series.loc[all_series['Exhaled TV (patient)'] < 0 ,     'Exhaled TV (patient)'] = np.nan
    all_series.loc[all_series['Exhaled TV (patient)'] > 1200 ,  'Exhaled TV (patient)'] = np.nan
    all_series.loc[all_series['Exhaled TV (machine)'] < 0 ,     'Exhaled TV (machine)'] = np.nan
    all_series.loc[all_series['Exhaled TV (machine)'] > 1200 ,  'Exhaled TV (machine)'] = np.nan
    all_series.loc[all_series['Peak Pressure'] < 0 ,   'Peak Pressure'] = np.nan
    all_series.loc[all_series['Peak Pressure'] > 60 ,  'Peak Pressure'] = np.nan
    all_series.loc[all_series['Plateau Pressure'] < 0 ,    'Plateau Pressure'] = np.nan
    all_series.loc[all_series['Plateau Pressure'] > 80 ,   'Plateau Pressure'] = np.nan
    all_series.loc[all_series['Peak Insp. Pressure'] < 0 ,    'Peak Insp. Pressure'] = np.nan
    all_series.loc[all_series['Peak Insp. Pressure'] > 80 ,   'Peak Insp. Pressure'] = np.nan
    all_series.loc[all_series['Mean Airway Pressure'] < 0 ,    'Mean Airway Pressure'] = np.nan
    all_series.loc[all_series['Mean Airway Pressure'] > 60 ,   'Mean Airway Pressure'] = np.nan
    all_series.loc[all_series['Inspiratory Flow Rate'] < 0 ,    'Inspiratory Flow Rate'] = np.nan
    all_series.loc[all_series['Inspiratory Flow Rate'] > 105 ,  'Inspiratory Flow Rate'] = np.nan
    all_series.loc[all_series['O2 Percentage'] < 0 ,    'O2 Percentage'] = np.nan
    all_series.loc[all_series['O2 Percentage'] > 105 ,  'O2 Percentage'] = np.nan
    all_series.loc[all_series['Oxygen Flow Rate'] < 0 ,   'Oxygen Flow Rate'] = np.nan
    all_series.loc[all_series['Oxygen Flow Rate'] > 50 ,  'Oxygen Flow Rate'] = np.nan
    all_series.loc[all_series['FiO2 (Set)'] < 0 ,    'FiO2 (Set)'] = np.nan
    all_series.loc[all_series['FiO2 (Set)'] > 105 ,  'FiO2 (Set)'] = np.nan
    all_series.loc[all_series['Pressure Support (Set)'] < 0 ,    'Pressure Support (Set)'] = np.nan
    all_series.loc[all_series['Pressure Support (Set)'] > 60 ,   'Pressure Support (Set)'] = np.nan
    all_series.loc[all_series['PEEP (Set)'] < 0 ,    'PEEP (Set)'] = np.nan
    all_series.loc[all_series['PEEP (Set)'] > 40 ,   'PEEP (Set)'] = np.nan
    all_series.loc[all_series['LPM O2 (Set)'] < 0 ,    'LPM O2 (Set)'] = np.nan
    all_series.loc[all_series['LPM O2 (Set)'] > 105 ,  'LPM O2 (Set)'] = np.nan
    all_series.loc[all_series['Vent Rate (Set)'] < 0 ,    'Vent Rate (Set)'] = np.nan
    all_series.loc[all_series['Vent Rate (Set)'] > 70 ,   'Vent Rate (Set)'] = np.nan
    all_series.loc[all_series['Tidal Volume (Set)'] < 0 ,     'Tidal Volume (Set)'] = np.nan
    all_series.loc[all_series['Tidal Volume (Set)'] > 1000 ,  'Tidal Volume (Set)'] = np.nan
    all_series.loc[all_series['TV/kg IBW (Set)'] < 0 ,   'TV/kg IBW (Set)'] = np.nan
    all_series.loc[all_series['TV/kg IBW (Set)'] > 20 ,  'TV/kg IBW (Set)'] = np.nan
    all_series.loc[all_series['PEEP/CPAP (Set)'] < 0 ,   'PEEP/CPAP (Set)'] = np.nan
    all_series.loc[all_series['PEEP/CPAP (Set)'] > 25 ,  'PEEP/CPAP (Set)'] = np.nan
    all_series.loc[all_series['Flow Sensitivity (Set)'] < 0 ,   'Flow Sensitivity (Set)'] = np.nan
    all_series.loc[all_series['Flow Sensitivity (Set)'] > 10 ,  'Flow Sensitivity (Set)'] = np.nan
    all_series.loc[all_series['Peak Flow (Set)'] < 0 ,    'Peak Flow (Set)'] = np.nan
    all_series.loc[all_series['Peak Flow (Set)'] > 150 ,  'Peak Flow (Set)'] = np.nan
    all_series.loc[all_series['Pain Goal'] < 0 ,   'Pain Goal'] = np.nan
    all_series.loc[all_series['Pain Goal'] > 12 ,  'Pain Goal'] = np.nan
    all_series.loc[all_series['Pain Score'] < 0 ,   'Pain Score'] = np.nan
    all_series.loc[all_series['Pain Score'] > 12 ,  'Pain Score'] = np.nan
    all_series.loc[all_series['GCS Total'] < 0 ,   'GCS Total'] = np.nan
    all_series.loc[all_series['GCS Total'] > 20 ,  'GCS Total'] = np.nan
    all_series.loc[all_series['Motor'] < 0 ,  'Motor'] = np.nan
    all_series.loc[all_series['Motor'] > 8 ,  'Motor'] = np.nan
    all_series.loc[all_series['Verbal'] < 0 ,  'Verbal'] = np.nan
    all_series.loc[all_series['Verbal'] > 8 ,  'Verbal'] = np.nan
    all_series.loc[all_series['Eyes'] < 0 ,  'Eyes'] = np.nan
    all_series.loc[all_series['Eyes'] > 8 ,  'Eyes'] = np.nan
    all_series.loc[all_series['RASS'] < -7 ,  'RASS'] = np.nan
    all_series.loc[all_series['RASS'] > 7 ,   'RASS'] = np.nan
    all_series.loc[all_series['Fall Risk'] < 0 ,  'Fall Risk'] = np.nan
    all_series.loc[all_series['Fall Risk'] > 5 ,  'Fall Risk'] = np.nan
    all_series.loc[all_series['Delirium Score'] < 0 ,   'Delirium Score'] = np.nan
    all_series.loc[all_series['Delirium Score'] > 10 ,  'Delirium Score'] = np.nan
    all_series.loc[all_series['Symptoms of Delirium Present'] < 0 ,  'Symptoms of Delirium Present'] = np.nan
    all_series.loc[all_series['Symptoms of Delirium Present'] > 2 ,  'Symptoms of Delirium Present'] = np.nan
    all_series.loc[all_series['Sedation Goal'] < -8 ,   'Sedation Goal'] = np.nan
    all_series.loc[all_series['Sedation Goal'] > 8 ,    'Sedation Goal'] = np.nan
    all_series.loc[all_series['Sedation Score'] < -8 ,   'Sedation Score'] = np.nan
    all_series.loc[all_series['Sedation Score'] > 8 ,    'Sedation Score'] = np.nan
    all_series.loc[all_series['Ventilator Type'] < 0 ,   'Ventilator Type'] = np.nan
    all_series.loc[all_series['Ventilator Type'] > 10 ,  'Ventilator Type'] = np.nan
    all_series.loc[all_series['Pain Present'] < 0 ,  'Pain Present'] = np.nan
    all_series.loc[all_series['Pain Present'] > 2 ,  'Pain Present'] = np.nan
    all_series.loc[all_series['C-Reactive Protein'] < 0 ,    'C-Reactive Protein'] = np.nan
    all_series.loc[all_series['C-Reactive Protein'] > 400 ,  'C-Reactive Protein'] = np.nan
    all_series.loc[all_series['cortisol'] < 0 ,    'cortisol'] = np.nan
    all_series.loc[all_series['cortisol'] > 250 ,  'cortisol'] = np.nan
    all_series.loc[all_series['ST1'] < -15 ,  'ST1'] = np.nan
    all_series.loc[all_series['ST1'] > 15  ,  'ST1'] = np.nan
    all_series.loc[all_series['ST2'] < -15 ,  'ST2'] = np.nan
    all_series.loc[all_series['ST2'] > 15  ,  'ST2'] = np.nan
    all_series.loc[all_series['ST3'] < -15 ,  'ST3'] = np.nan
    all_series.loc[all_series['ST3'] > 15  ,  'ST3'] = np.nan
        
    return all_series

### Convert data to timeseries

In [7]:
def convert_events_to_timeseries(all_tables, all_variables):

    metadata = all_tables[['itemoffset', 'patientunitstayid']].sort_values(by=['itemoffset'])\
                    .drop_duplicates(keep='first').set_index('itemoffset')

    timeserie = all_tables[['itemoffset', 'itemname', 'itemvalue']]\
                    .sort_values(by=['itemoffset'], axis=0)\
                    .drop_duplicates(subset=['itemoffset', 'itemname'], keep='last')

    time_piv = timeserie.pivot(index='itemoffset', columns='itemname', values='itemvalue')
    timeseries = time_piv.merge(metadata, left_index=True, right_index=True).sort_index(axis=0).reset_index()

    missing_vars = [v for v in all_variables if v not in timeseries.columns]
    missing_dataframes = [pd.DataFrame({v: np.nan}, index=timeseries.index) for v in missing_vars]

    if missing_dataframes:
        timeseries = pd.concat([timeseries] + missing_dataframes, axis=1)

    return timeseries

### Fix varibales

In [8]:
def convert_variables(df):
    
    if df.admissionweight.isnull().all():
        if ~ df.Bodyweight.isnull().all():
            first_weight = df.loc[~df.Bodyweight.isnull()].Bodyweight.iloc[0]
            df['admissionweight'] = first_weight
            df["admissionweight"] = df["admissionweight"].round(1)
        
    df.loc[df['GCS Total'].isnull(), 'GCS Total'] = df['Motor'] + df['Verbal'] + df['Eyes']
    df["Temperature (C)"] = df["Temperature (C)"].round(1)
    df.drop(columns=['Bodyweight'], inplace=True)
    
    return df

### Seperate Vital Sign Variables

In [9]:
def vital_sign_frame(df):
    
    vital_columns = ['uniquepid', 'patienthealthsystemstayid', 'patientunitstayid', 'gender', 'age', 'ethnicity', 
                     'itemoffset', 'Heart Rate', 'Respiratory Rate', 'Temperature (C)', 'O2 Saturation', 'SpO2',
                     'Invasive BP Mean', 'Invasive BP Diastolic', 'Invasive BP Systolic', 
                     'Non-Invasive BP Mean', 'Non-Invasive BP Diastolic', 'Non-Invasive BP Systolic',
                     'ST1', 'ST2', 'ST3', 
                     'lactate', 'Base Excess', 'FiO2', 'FiO2 (Set)', 'paO2', 'paCO2', 'ETCO2', 'PEEP', 'PEEP (Set)', 
                     'Tidal Volume', 'Tidal Volume (Set)', 'pH', 'Urine_IO', 'Oxygen Flow Rate', 'Ventilator Type', 
                     'Vasopressin_PRC', 'Epinephrine_PRC', 'Milrinone_PRC', 'Norepinephrine_PRC', 'Phenylephrine_PRC', 'Dobutamine_PRC', 
                     'Vassopressin_IO', 'Epinephrine_IO',  'Milrinone_IO',  'Norepinephrine_IO',  'Phenylephrine_IO',  'Dobutamine_IO',  
                      ]
    
    vital_df = df[vital_columns]
    vital_df = vital_df.drop_duplicates()
    vital_df = vital_df.dropna(thresh=8, axis=0)
    vital_df = vital_df.reset_index(drop=True).sort_values(by= 'itemoffset')
    
    return vital_df

### Create hourly based data frames

In [10]:
def create_template_all_frame(all_series):
    
    id_los = all_series.groupby('patientunitstayid')[['patientunitstayid', 'unitdischargeoffset']].head(1)
    id_los['icuLos_h']  = (id_los['unitdischargeoffset']/60).apply(np.ceil) + 1

    stayids = list(id_los.patientunitstayid.unique())
    template_list = []

    for id in stayids:
        df = {'Bins': range(0, 1 + id_los[id_los.patientunitstayid == id].icuLos_h.values[0].astype(int)) }
        df = pd.DataFrame(df)
        df['patientunitstayid'] = id
        template_list.append(df)

    template_df = pd.concat(template_list)
    
    return template_df

In [11]:
def create_template_vital_frame(all_series, bt_vital=5):
    
    id_los = all_series.groupby('patientunitstayid')[['patientunitstayid', 'unitdischargeoffset']].head(1)
    id_los['icuLos_h']  = (id_los['unitdischargeoffset']/60).apply(np.ceil) + 1

    stayids = list(id_los.patientunitstayid.unique())
    template_list = []

    for id in stayids:
        df = {'Bins': range(0, (1 + id_los[id_los.patientunitstayid == id].icuLos_h.values[0].astype(int)) * 60, bt_vital) }
        df = pd.DataFrame(df)
        df['patientunitstayid'] = id
        template_list.append(df)

    template_df = pd.concat(template_list)
    
    return template_df

### Calculate bins

In [12]:
def bin_time(frame_df, frame_vital, bt_df=60, bt_vital=5):

    frame_df['Bins'] = (frame_df['itemoffset']/bt_df).astype(int)
    frame_vital['Bins'] = (frame_vital['itemoffset']/bt_vital).astype(int) * bt_vital

    return frame_df, frame_vital

### Seperating categorical, continuse, static variables

In [13]:
def seperate_varibale_types(all_series):
    
    static_columns = ['uniquepid', 'patienthealthsystemstayid', 'patientunitstayid', 'gender', 'age', 'ethnicity', 
                      'admissionheight', 'admissionweight', 'apacheadmissiondx', 'unitdischargeoffset', 
                      'hospitaldischargeoffset', 'unitdischargestatus', 'hospitaldischargestatus', 
                      'numbedscategory', 'teachingstatus', 'apacheadmissioncategory']
    
    categorical_columns = ['patientunitstayid', 'Bins', 
                           'ICD-9', 'ICD-10', 'ICD-10_Embedding', 'pastHistory', 'elixhauser_comorbidity', 
                           'Verbal', 'Motor', 'Eyes', 'GCS Total', 'Pain Score', 'Pain Present', 'Pain Goal', 
                           'Sedation Score', 'Sedation Goal', 'Symptoms of Delirium Present', 'Delirium Score', 
                           'Delirium Scale', 'Fall Risk', 'RASS', 'O2 Admin Device', 'Ventilator Type',
                           'Fentanyl_PRC', 'Propofol_PRC', 'Norepinephrine_PRC', 'Insulin_PRC',
                           'Midazolam_PRC', 'Heparin_PRC', 'Dexmedetomidine_PRC', 'Amiodarone_PRC', 
                           'Vasopressin_PRC', 'Phenylephrine_PRC', 'Dopamine_PRC', 'Nicardipine_PRC',
                           'Milrinone_PRC', 'Pantoprazole_PRC', 'Diltiazem_PRC', 'Dobutamine_PRC', 
                           'Nitroglycerin_PRC', 'Epinephrine_PRC','Antibiotic_PRC', 'Warfarin_PRC', 'Vasopressors']

    all_columns = list(all_series.columns)
    continuse_columns = (set(all_columns) - set(categorical_columns)) - set(static_columns)
    continuse_columns = list(continuse_columns)
    continuse_columns.remove('itemoffset')
    continuse_columns.extend(['patientunitstayid', 'Bins'])
    
    return static_columns, categorical_columns, continuse_columns

### Seperate dataframes based on variables type

In [14]:
def seperate_frame_types(all_series, static_columns, categorical_columns, continuse_columns):
    
    static_df = all_series[static_columns]
    static_df = static_df.drop_duplicates()

    categorical_df = all_series[categorical_columns]
    categorical_df = categorical_df.drop_duplicates()
    categorical_df = categorical_df.dropna(thresh=3, axis=0)

    dynamic_df = all_series[continuse_columns]
    dynamic_df = dynamic_df.drop_duplicates()
    dynamic_df = dynamic_df.dropna(thresh=3, axis=0)
    
    return static_df, categorical_df, dynamic_df

### Binning data

In [15]:
def binning(template_df, categorical_df, dynamic_df, categorical_columns, continuse_columns):
    
    cat_columns = [col for col in categorical_columns if col not in ('patientunitstayid', 'Bins')]
    cont_column = [col for col in continuse_columns   if col not in ('patientunitstayid', 'Bins')]

    for category in cat_columns:
        temp_df = categorical_df.groupby(['patientunitstayid', 'Bins'], group_keys=True)[category].apply(pd.Series.mode).reset_index()[['patientunitstayid', 'Bins', category]]
        if category not in ['ICD-9', 'ICD-10',  'ICD-10_Embedding', 'pastHistory', 'elixhauser_comorbidity']:
            temp_df.drop_duplicates(['patientunitstayid', 'Bins'], inplace=True)
        template_df = pd.merge(template_df, temp_df, on=['patientunitstayid', 'Bins'], how='left')

    for continuse in cont_column:
        temp_df = dynamic_df.groupby(['patientunitstayid', 'Bins'], group_keys=True)[continuse].apply(pd.Series.mean).reset_index()[['patientunitstayid', 'Bins', continuse]]
        temp_df[continuse] = round(temp_df[continuse], 1)
        template_df = pd.merge(template_df, temp_df, on=['patientunitstayid', 'Bins'], how='left')
        
    return template_df

In [16]:
def binning_vital(template_vital_df, df_bin_vital):

    cont_column = ['Heart Rate', 'Respiratory Rate', 'Temperature (C)', 'O2 Saturation', 'SpO2',
                   'Invasive BP Mean', 'Invasive BP Diastolic', 'Invasive BP Systolic', 
                   'Non-Invasive BP Mean', 'Non-Invasive BP Diastolic', 'Non-Invasive BP Systolic',
                   'ST1', 'ST2', 'ST3',
                   'lactate', 'Base Excess', 'FiO2', 'FiO2 (Set)', 'paO2', 'paCO2', 'ETCO2', 'PEEP', 'PEEP (Set)', 
                   'Tidal Volume', 'Tidal Volume (Set)', 'pH', 'Urine_IO', 'Oxygen Flow Rate', 'Ventilator Type', 
                   'Vasopressin_PRC', 'Epinephrine_PRC', 'Milrinone_PRC', 'Norepinephrine_PRC', 'Phenylephrine_PRC', 'Dobutamine_PRC', 
                   'Vassopressin_IO', 'Epinephrine_IO',  'Milrinone_IO',  'Norepinephrine_IO',  'Phenylephrine_IO',  'Dobutamine_IO']

    for continuse in cont_column:
        temp_df = df_bin_vital.groupby(['patientunitstayid', 'Bins'], group_keys=True)[continuse].apply(pd.Series.mean).reset_index()[['patientunitstayid', 'Bins', continuse]]
        temp_df[continuse] = round(temp_df[continuse], 1)
        template_vital_df = pd.merge(template_vital_df, temp_df, on=['patientunitstayid', 'Bins'], how='left')
        
    return template_vital_df

In [17]:
admission  = dataframe_from_csv(os.path.join(path_timeseries, '141196', 'admission.csv'))
all_tables = dataframe_from_csv(os.path.join(path_timeseries, '141196', 'all_tables.csv'))

### Extract timeseries data based on ICU - multiprocessing

In [17]:
def process_csv(stay_dir):
    
    dn = os.path.join(path_timeseries, stay_dir)
        
    try:
        sys.stdout.flush()

        admission  = dataframe_from_csv(os.path.join(path_timeseries, stay_dir, 'admission.csv'))
        all_tables = dataframe_from_csv(os.path.join(path_timeseries, stay_dir, 'all_tables.csv'))

        all_tables = filter_on_variabels(all_tables, all_variables)
        all_tables = all_tables.drop_duplicates()
        all_tables = all_tables.sort_values(by=['itemoffset'])

        timeepisode = convert_events_to_timeseries(all_tables, all_variables)
        timeepisode = outlier_removal(timeepisode)

        merged_df = pd.merge(timeepisode, admission, on='patientunitstayid')
        merged_df = merged_df[merged_df.itemoffset < (admission.unitdischargeoffset.values[0] + 120)]
        merged_df = merged_df.sort_values(by=['itemoffset'])
        merged_df = convert_variables(merged_df)

        vital_df = vital_sign_frame(merged_df)

        template_df = create_template_all_frame(merged_df)
        template_vital_df = create_template_vital_frame(merged_df)
        df_bin, df_bin_vital = bin_time(merged_df, vital_df)

        static_columns, categorical_columns, continuse_columns = seperate_varibale_types(df_bin)
        static_df, categorical_df, dynamic_df = seperate_frame_types(df_bin, static_columns, categorical_columns, continuse_columns)

        binned_df = binning(template_df, categorical_df, dynamic_df, categorical_columns, continuse_columns)
        binned_vital_df = binning_vital(template_vital_df, df_bin_vital)

        final = pd.merge(binned_df, static_df, on='patientunitstayid', how='left')
        final = final.reset_index(drop=True)
        binned_vital_df = binned_vital_df.reset_index(drop=True)

        final.to_csv(os.path.join(dn, 'raw_timeseries.csv'), index=False)
        binned_vital_df.to_csv(os.path.join(dn, 'raw_vital_timeseries.csv'), index=False)
        
    except Exception as e:
        print(f"Error processing {stay_dir}: {e}")
        exception_stayID.append(stay_dir)

In [18]:
dirs = os.listdir(path_timeseries)

exception_stayID = []
num_processes = cpu_count()

In [ ]:
with Pool(num_processes) as p:
    for _ in tqdm(p.imap(process_csv, dirs), total=len(dirs)):
        pass

In [21]:
if exception_stayID:
    with open("../Data/Cohort/exception_stayID_raw_data.txt", "w") as f:
        for subject_id in exception_stayID:
            f.write(str(subject_id) +"\n")

### Extract timeseries data based on ICU - For Loop

In [ ]:
def extract_time_series_from_subject(output_path, all_variables):
    
    dirs = os.listdir(output_path)
    total_dirs = len(dirs)
    processed_dirs = 0
    last_printed_progress = -1
    
    exception_stayID = []
    
    for stay_dir in dirs:
        dn = os.path.join(output_path, stay_dir)
        
        try:
            sys.stdout.flush()
            
            admission  = dataframe_from_csv(os.path.join(output_path, stay_dir, 'admission.csv'))
            all_tables = dataframe_from_csv(os.path.join(output_path, stay_dir, 'all_tables.csv'))

            all_tables = filter_on_variabels(all_tables, all_variables)
            all_tables = all_tables.drop_duplicates()
            all_tables = all_tables.sort_values(by=['itemoffset'])

            timeepisode = convert_events_to_timeseries(all_tables, all_variables)
            timeepisode = outlier_removal(timeepisode)

            merged_df = pd.merge(timeepisode, admission, on='patientunitstayid')
            merged_df = merged_df[merged_df.itemoffset < (admission.unitdischargeoffset.values[0] + 120)]
            merged_df = merged_df.sort_values(by=['itemoffset'])
            merged_df = convert_variables(merged_df)

            vital_df = vital_sign_frame(merged_df)

            template_df = create_template_all_frame(merged_df)
            template_vital_df = create_template_vital_frame(merged_df)
            df_bin, df_bin_vital = bin_time(merged_df, vital_df)

            static_columns, categorical_columns, continuse_columns = seperate_varibale_types(df_bin)
            static_df, categorical_df, dynamic_df = seperate_frame_types(df_bin, static_columns, categorical_columns, continuse_columns)

            binned_df = binning(template_df, categorical_df, dynamic_df, categorical_columns, continuse_columns)
            binned_vital_df = binning_vital(template_vital_df, df_bin_vital)

            final = pd.merge(binned_df, static_df, on='patientunitstayid', how='left')
            final = final.reset_index(drop=True)
            binned_vital_df = binned_vital_df.reset_index(drop=True)

            final.to_csv(os.path.join(dn, 'raw_timeseries.csv'), index=False)
            binned_vital_df.to_csv(os.path.join(dn, 'raw_vital_timeseries.csv'), index=False)
            
            processed_dirs += 1
            progress_percentage = (processed_dirs / total_dirs) * 100
            
            if int(progress_percentage) > last_printed_progress:
                last_printed_progress = int(progress_percentage)
                sys.stdout.write(f"\rProgress: {progress_percentage:.2f}% ({processed_dirs}/{total_dirs}) - Processing StayID {stay_dir}...\n")
                sys.stdout.flush()

        except :
            exception_stayID.append(stay_dir)
            continue
            
    print('DONE')
    
    with open("../Data/Cohort/exception_stayID.txt", "w") as f:
        for subject_id in exception_stayID:
            f.write(str(subject_id) +"\n")

In [ ]:
# extract_time_series_from_subject(path_timeseries, all_variables)